# GTEx CLAMP models with BP prior - Multiple Seeds

**Environment:** `clamp-analyses`  

This notebook builds 3 CLAMP models using the BP (GO Biological Process) prior on GTEx data, each with a different random seed.

Steps (repeated for each seed):
1. Load preprocessed GTEx data (FBM, genes, samples)
2. Compute SVD with current seed
3. Estimate CLAMP K from SVD
4. Run CLAMPbase
5. Run CLAMPfull with BP prior

Each run uses a different seed (`base_seed + 0:2`) for reproducibility, allowing comparison of model stability across different initializations.

## Load libraries

In [6]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(CLAMP)

source(here("config.R"))

## Configuration

In [7]:
# Base output directory
output_data_dir <- config$GTEx$OUTPUT_DIR
dir.create(output_data_dir, showWarnings = FALSE, recursive = TRUE)

# SVD parameters
N_CORES <- config$GTEx$N_CORES

# Seeds for reproducibility - run 3 models with different seeds
base_seed <- config$GTEx$RANDOM_SVD_SEED
seeds <- base_seed + 0:2
n_runs <- length(seeds)
message("Will run ", n_runs, " models with seeds: ", paste(seeds, collapse = ", "))

Will run 3 models with seeds: 123, 124, 125



## Load preprocessed GTEx data

In [8]:
# Load preprocessed GTEx FBM
gtex_fbm_filt <- readRDS(file.path(output_data_dir, "gtex_fbm_filt.rds"))

# Load genes and samples
gtex_genes <- readRDS(file.path(output_data_dir, "gtex_genes.rds"))
samples <- readRDS(file.path(output_data_dir, "gtex_samples.rds"))

n_genes <- nrow(gtex_fbm_filt)
n_samples <- ncol(gtex_fbm_filt)

message("Loaded GTEx data with ", n_genes, " genes and ", n_samples, " samples")

Loaded GTEx data with 21613 genes and 17382 samples



## Load BP pathway prior

In [9]:
# Load BP (GO Biological Process) prior
BP_gmt <- getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GO_Biological_Process_2025")
BP_gmtList <- list(BP = BP_gmt)

# Prefix gene-set names with library name
names(BP_gmtList[["BP"]]) <- paste0("BP_", names(BP_gmtList[["BP"]]))

BP_pathMat <- gmtListToSparseMat(BP_gmtList)
BP_matched <- getMatchedPathwayMat(BP_pathMat, gtex_genes)

message("Loaded and matched BP pathway matrix")

Auto-detected name: GO_Biological_Process_2025

Using cached file for GO_Biological_Process_2025

There are 12116 genes in the intersection between data and prior

Removing 2020 pathways

Loaded and matched BP pathway matrix



## Run 3 CLAMP models with different seeds

In [10]:
# Store results summary
results_summary <- data.frame(
  run = integer(),
  seed = integer(),
  n_samples = integer(),
  n_genes = integer(),
  CLAMP_K = integer(),
  stringsAsFactors = FALSE
)

# SVD K
SVD_K <- min(n_genes, n_samples) - 1
message("SVD K = ", SVD_K)

for (run_idx in seq_len(n_runs)) {
  current_seed <- seeds[run_idx]
  message("\n", strrep("=", 60))
  message("RUN ", run_idx, "/", n_runs, " - Seed: ", current_seed)
  message(strrep("=", 60))
  
  # Set seed for reproducibility
  set.seed(current_seed)
  
  # Create output directory for this run
  output_dir <- file.path(output_data_dir, paste0("bp_coverage_seed_", run_idx))
  dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)
  
  # Compute SVD with current seed
  message("Computing SVD...")
  
  if (N_CORES > 1) {
    options(bigstatsr.check.parallel.blas = FALSE)
    blas_nproc <- getOption("default.nproc.blas")
    options(default.nproc.blas = NULL)
  }
  
  svd_result <- big_randomSVD(gtex_fbm_filt, k = SVD_K, ncores = N_CORES)
  
  if (N_CORES > 1) {
    options(bigstatsr.check.parallel.blas = TRUE)
    options(default.nproc.blas = blas_nproc)
  }
  
  # Remove NaN values from SVD
  valid_idx <- which(!is.nan(svd_result$d))
  svd_result$d <- svd_result$d[valid_idx]
  svd_result$u <- svd_result$u[, valid_idx, drop = FALSE]
  svd_result$v <- svd_result$v[, valid_idx, drop = FALSE]
  
  saveRDS(svd_result, file = file.path(output_dir, "svd.rds"))
  
  # Estimate CLAMP K
  svd_list <- list(d = svd_result$d)
  CLAMP_K <- num.pc(svd_list) * 2
  message("CLAMP K = ", CLAMP_K)
  saveRDS(CLAMP_K, file = file.path(output_dir, "CLAMP_K.rds"))
  
  # Save run info
  saveRDS(list(
    run = run_idx,
    seed = current_seed,
    n_samples = n_samples,
    n_genes = n_genes,
    CLAMP_K = CLAMP_K
  ), file = file.path(output_dir, "run_info.rds"))
  
  # CLAMPbase
  message("Running CLAMPbase...")
  baseRes <- CLAMPbase(
    Y = gtex_fbm_filt,
    svdres = svd_result,
    trace = TRUE,
    clamp_k = CLAMP_K
  )
  
  baseRes$Z <- data.frame(baseRes$Z)
  rownames(baseRes$Z) <- gtex_genes
  baseRes$B <- data.frame(baseRes$B)
  colnames(baseRes$B) <- samples
  
  saveRDS(baseRes, file = file.path(output_dir, "CLAMPbase.rds"))
  
  model_dir <- file.path(output_dir, "CLAMPbase")
  dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
  write.csv(baseRes$B, file.path(model_dir, "B.csv"))
  write.csv(baseRes$Z, file.path(model_dir, "Z.csv"))
  
  # CLAMPfull with BP prior
  message("Running CLAMPfull with BP prior...")
  fullRes <- CLAMPfull(
    Y = gtex_fbm_filt,
    svdres = svd_result,
    priorMat = BP_matched,
    clamp.base.result = baseRes,
    use_cpp = TRUE,
    trace = TRUE,
    clamp_k = CLAMP_K
  )
  
  fullRes$Z <- data.frame(fullRes$Z)
  rownames(fullRes$Z) <- gtex_genes
  fullRes$B <- data.frame(fullRes$B)
  colnames(fullRes$B) <- samples
  fullRes$summary <- fullRes$summary %>%
    dplyr::rename(LV = LV_index) %>%
    dplyr::mutate(LV = paste0('LV', LV))
  
  saveRDS(fullRes, file = file.path(output_dir, "CLAMPfull_BP.rds"))
  
  model_dir <- file.path(output_dir, "CLAMPfull_BP")
  dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)
  write.csv(fullRes$B, file.path(model_dir, "B.csv"))
  write.csv(fullRes$Z, file.path(model_dir, "Z.csv"))
  write.csv(fullRes$summary, file.path(model_dir, "summary.csv"))
  
  # Store summary
  results_summary <- rbind(results_summary, data.frame(
    run = run_idx,
    seed = current_seed,
    n_samples = n_samples,
    n_genes = n_genes,
    CLAMP_K = CLAMP_K
  ))
  
  # Clean up memory
  rm(svd_result, baseRes, fullRes)
  gc()
}

message("\n", strrep("=", 60))
message("All ", n_runs, " runs completed!")
message(strrep("=", 60))

SVD K = 17381



RUN 1/3 - Seed: 123


Computing SVD...

CLAMP K = 412

Running CLAMPbase...

****

CLAMP k is set to 412

L1 is set to 45.2778145717362

L2 is set to 135.833443715208

Progress 1 / 200 | Bdiff=0.222227, minCor=0.616286

Progress 2 / 200 | Bdiff=0.030933, minCor=0.915387

Progress 3 / 200 | Bdiff=0.014314, minCor=0.958797

Progress 4 / 200 | Bdiff=0.009708, minCor=0.976381

Progress 5 / 200 | Bdiff=0.007364, minCor=0.983586

Progress 6 / 200 | Bdiff=0.005958, minCor=0.987160

Progress 7 / 200 | Bdiff=0.005016, minCor=0.988719

Progress 8 / 200 | Bdiff=0.004328, minCor=0.989041

Progress 9 / 200 | Bdiff=0.003798, minCor=0.989755

Progress 10 / 200 | Bdiff=0.003379, minCor=0.990697

Progress 11 / 200 | Bdiff=0.003044, minCor=0.992363

Progress 12 / 200 | Bdiff=0.002777, minCor=0.994038

Progress 13 / 200 | Bdiff=0.002561, minCor=0.994884

Progress 14 / 200 | Bdiff=0.002384, minCor=0.995539

Progress 15 / 200 | Bdiff=0.002234, minCor=0.995313

Progress 16 / 200 | Bdiff=0.0

## Summary

In [11]:
# Display results summary
message("\nResults Summary:")
print(kable(results_summary, format = "simple"))

# Save summary
saveRDS(results_summary, file = file.path(output_data_dir, "bp_coverage_gtex_summary.rds"))
write.csv(results_summary, file.path(output_data_dir, "bp_coverage_gtex_summary.csv"), row.names = FALSE)


Results Summary:





 run   seed   n_samples   n_genes   CLAMP_K
----  -----  ----------  --------  --------
   1    123       17382     21613       412
   2    124       17382     21613       412
   3    125       17382     21613       412
